# Lab: Word-Level Text Generation with RNNs

## 1. Introduction: From Characters to Words

In the previous lab, we predicted the **next character** (e.g., 'D' -> 'e' -> 'e' -> 'p').
However, meaningful language understanding happens at the **Word Level**.

In this lab, we will:
1.  **Tokenize** text into words (instead of characters).
2.  **Build a Vocabulary** of unique words.
3.  **Train an LSTM** to predict the next word in a sentence.
4.  **Generate** new sentences (e.g., "Deep learning is a field...").



In [ ]:
import torch
import torch.nn as nn
import numpy as np
import re
import random

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 2. Data Processing: Word Tokenization

We use the same "Deep Learning" text, but now we split it into **words**.
We will also perform basic cleaning (remove some punctuation or treat it as tokens).



In [ ]:
text_data = """
Deep learning is part of a broader family of machine learning methods based on artificial neural networks with representation learning. 
Learning can be supervised, semi-supervised or unsupervised. 
Deep-learning architectures such as deep neural networks, deep belief networks, deep reinforcement learning, recurrent neural networks, 
convolutional neural networks and transformers have been applied to fields including computer vision, speech recognition, natural language processing, machine translation, bioinformatics, drug design, medical image analysis, climate science, material inspection and board game programs, where they have produced results comparable to and in some cases surpassing human expert performance. 
Artificial neural networks (ANNs) were inspired by information processing and distributed communication nodes in biological systems. 
ANNs have various differences from biological brains. Specifically, artificial neural networks tend to be static and symbolic, while the biological brain of most living organisms is dynamic (plastic) and analog.
"""

# 1. Tokenization (Regex to split words and punctuation)
# \w+|[^\w\s] -> Matches words OR punctuation (keeps dots, commas as tokens)
def tokenize(text):
    text = text.lower() # Case insensitive
    return re.findall(r"\w+|[^\w\s]", text)

words = tokenize(text_data)
print(f"Total Words: {len(words)}")
print(f"Sample: {words[:10]}")


## 3. Vocabulary Building

Mapped each unique word to an index (Integer).



In [ ]:
# Create Vocabulary
vocab = sorted(list(set(words)))
vocab_size = len(vocab)

word_to_ix = { w:i for i,w in enumerate(vocab) }
ix_to_word = { i:w for i,w in enumerate(vocab) }

print(f"Vocabulary Size: {vocab_size} unique words")
print(f"Index for 'learning': {word_to_ix.get('learning', 'Not Found')}")


In [ ]:
def text_to_tensor(text_list):
    """Converts a list of words to a tensor of indices."""
    indices = [word_to_ix[w] for w in text_list]
    return torch.tensor(indices, dtype=torch.long).to(device)

def tensor_to_text(tensor):
    """Converts indices back to string."""
    if tensor.dim() == 2:
        tensor = tensor[0]
    return " ".join([ix_to_word[idx.item()] for idx in tensor])

# Test
test_seq = ["deep", "learning", "is"]
print(f"Tensor: {text_to_tensor(test_seq)}")


## 4. The Model (LSTM)

We reuse the SequenceModel architecture. Note that `input_size` is now the number of words in our vocab (~80+), not just 35 chars.



In [ ]:
class SequenceModel(nn.Module):
    def __init__(self, model_type, input_size, hidden_size, output_size, n_layers=1):
        super(SequenceModel, self).__init__()
        self.model_type = model_type.lower()
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        
        self.embedding = nn.Embedding(input_size, hidden_size)
        
        # LSTM
        self.rnn = nn.LSTM(hidden_size, hidden_size, n_layers, batch_first=True)
        
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x, hidden):
        # x: [Batch, Seq]
        embed = self.embedding(x)
        out, hidden = self.rnn(embed, hidden)
        
        out = out.reshape(-1, self.hidden_size)
        out = self.fc(out)
        return out, hidden
    
    def init_hidden(self, batch_size):
        weight = next(self.parameters()).data
        return (weight.new(self.n_layers, batch_size, self.hidden_size).zero_(),
                weight.new(self.n_layers, batch_size, self.hidden_size).zero_())


## 5. Training Loop

We train on sequences of words.
*   **Chunk Length**: 5 words.
*   **Target**: The sequence shifted by one word.



In [ ]:
def get_batch(chunk_len=5, batch_size=16):
    input_batch = []
    target_batch = []
    
    for _ in range(batch_size):
        # Random start
        start_idx = np.random.randint(0, len(words) - chunk_len - 1)
        chunk = words[start_idx : start_idx + chunk_len + 1]
        
        input_data = text_to_tensor(chunk[:-1])
        target_data = text_to_tensor(chunk[1:])
        
        input_batch.append(input_data)
        target_batch.append(target_data)
        
    return torch.stack(input_batch), torch.stack(target_batch)

# Hyperparameters
HIDDEN_SIZE = 128
N_LAYERS = 1
EPOCHS = 300

model = SequenceModel('lstm', vocab_size, HIDDEN_SIZE, vocab_size, N_LAYERS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

print("--- Training Word-Level LSTM ---")
loss_history = []

for epoch in range(1, EPOCHS + 1):
    hidden = model.init_hidden(16)
    inputs, targets = get_batch()
    
    model.zero_grad()
    
    # Detach hidden
    hidden = (hidden[0].detach(), hidden[1].detach())
    
    output, hidden = model(inputs, hidden)
    loss = criterion(output, targets.view(-1))
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")


## 6. Generating Word Sequences

We initialize the model with a starting word (or sequence), then predict the next word, append it, and repeat.



In [ ]:
def generate_word_sequence(model, start_text="deep learning", predict_len=20, temperature=0.8):
    model.eval()
    hidden = model.init_hidden(1)
    
    # Tokenize start text
    start_tokens = tokenize(start_text)
    input_seq = text_to_tensor(start_tokens).unsqueeze(0) # [1, Seq]
    
    # Warm up hidden state
    for i in range(len(start_tokens) - 1):
        _, hidden = model(input_seq[:, i].unsqueeze(1), hidden)
    
    # Start predicting
    inp = input_seq[:, -1].unsqueeze(1)
    generated_tokens = start_tokens.copy()
    
    for _ in range(predict_len):
        output, hidden = model(inp, hidden)
        
        # Sampling
        output_dist = output.data.view(-1).div(temperature).exp()
        top_i = torch.multinomial(output_dist, 1)[0]
        
        pred_word = ix_to_word[top_i.item()]
        generated_tokens.append(pred_word)
        
        inp = text_to_tensor([pred_word]).unsqueeze(0)
        
    # Join with spaces (naive)
    # A real detokenizer would handle punctuation better
    return " ".join(generated_tokens)

print("--- Generated Text ---")
print(generate_word_sequence(model, start_text="artificial neural", predict_len=20))
print(generate_word_sequence(model, start_text="machine learning", predict_len=20))


## Conclusion

Tune the hyperparameters to get the best possible performance.
Try different architectures.